# Dataset preparation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mary-Vadi/nlp-toolkit/blob/main/notebooks/02_dataset_preparation.ipynb)

End-to-end preparation of a text classification dataset. By default it builds a small synthetic dataset with typical problems (empty rows, near-duplicates, conflicting labels). Set `SOURCE` to use your own CSV or a Hugging Face dataset instead.

The synthetic data is template-based and trivially separable, so baseline scores on it are not meaningful.

In [ ]:
import os, sys, subprocess

try:
    import nlp_toolkit
except ImportError:
    if os.path.isdir("../src/nlp_toolkit"):  # local clone
        sys.path.insert(0, os.path.abspath("../src"))
    else:  # Colab
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/Mary-Vadi/nlp-toolkit.git", "langdetect"], check=True)
    import nlp_toolkit

nlp_toolkit.__version__

## Load

In [ ]:
import random
import pandas as pd
import nlp_toolkit.dataset as D

SOURCE = "demo"  # "demo" | "csv" | "hf"

if SOURCE == "csv":
    df = D.load_text_data("my_data.csv")  # needs columns: text, label
elif SOURCE == "hf":
    from datasets import load_dataset
    df = load_dataset("rotten_tomatoes", split="train").to_pandas()
else:
    random.seed(0)
    pos = ["great", "excellent", "really useful", "lovely", "fast", "well made"]
    neg = ["terrible", "broken", "slow", "useless", "awful", "badly made"]
    items = ["phone", "laptop", "book", "course", "app", "camera", "keyboard"]
    noise = ["", " https://shop.example.com/item", " <br>", " &amp; more", " !!!", " @seller"]
    rows = []
    for _ in range(600):
        y = random.randint(0, 1)
        text = (f"The {random.choice(items)} is {random.choice(pos if y else neg)}{random.choice(noise)}. "
                f"{'I recommend it' if y else 'Do not buy it'}{random.choice(['.', '!', '...'])}")
        rows.append((text, y))
    rows += [("", 1), (None, 0), ("   ", 1), ("ok", 0), ("meh", 1)]
    rows += [(rows[i][0].upper(), rows[i][1]) for i in range(20)]
    rows += [(rows[i][0], 1 - rows[i][1]) for i in range(20, 25)]
    df = pd.DataFrame(rows, columns=["text", "label"]).sample(frac=1, random_state=0).reset_index(drop=True)

print(df.shape)
df.head()

## Drop empty rows

In [ ]:
df = D.drop_empty(df)
df.shape

## Clean

The original text is kept in `text_raw`.

In [ ]:
from nlp_toolkit import CleaningPipeline

PRESET = "transformer"
df = df.rename(columns={"text": "text_raw"})
df = D.clean_column(df, CleaningPipeline.from_preset(PRESET), text_col="text_raw", new_col="text")
df[["text_raw", "text"]].head()

## Label conflicts

Handle these before de-duplication; otherwise one arbitrary copy is kept.

In [ ]:
conflicts = D.find_label_conflicts(df)
print(len(conflicts), "rows in conflict")
conflicts[["text", "label"]].head(6)

In [ ]:
df = D.drop_label_conflicts(df)

## De-duplicate

`mode="normalized"` ignores case, whitespace and punctuation.

In [ ]:
df = D.remove_duplicates(df, mode="normalized")

## Length filter

In [ ]:
df = D.filter_by_length(df, min_words=3, max_words=512)

## Language filter

In [ ]:
df = D.detect_language(df)
print(df["lang"].value_counts().head())
df = df[df["lang"] == "en"].drop(columns="lang").reset_index(drop=True)

## Label distribution

In [ ]:
D.check_labels(df)

## EDA

Large length differences between classes can act as a shortcut feature.

In [ ]:
from nlp_toolkit import eda

eda.summary(df)

In [ ]:
eda.plot_length_distribution(df, unit="n_words");

In [ ]:
eda.plot_top_ngrams(df[df.label == 1]["text"], n=2, k=10, title="label 1: top bigrams")
eda.plot_top_ngrams(df[df.label == 0]["text"], n=2, k=10, title="label 0: top bigrams");

## Split and leakage check

In [ ]:
splits = D.split_dataset(df, val_size=0.1, test_size=0.1, seed=42)
for name, part in splits.items():
    print(f"{name:5} n={len(part):4}  pos_rate={part.label.mean():.3f}")
D.check_leakage(splits)

## Save

In [ ]:
D.save_splits(splits, "processed", fmt="jsonl")
sorted(os.listdir("processed"))

## Baseline

In [ ]:
from nlp_toolkit.features import baseline_classifier, top_features

model, report = baseline_classifier(splits["train"], splits["test"])
top_features(model, k=10)